In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
##  import pip

##  -U is for upgrade
##  -qqq is for quiet

##  !pip install -U -qqq git+https://github.com/qdosgit4/30040-experiments.git#subdirectory=experiment_5_reparameterisation_reimplementation

##  !pip show -f reparam

In [ ]:
import argparse, time, os, lzma, io
from datetime import datetime
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, Subset
from torchvision import datasets
from torchvision.transforms import ToTensor
import torch.nn.functional as F
import torch.distributions as dist

import matplotlib.pyplot as plt

##  from reparam.linear_bayesian import Linear_bayesian
##  from reparam.mini_model_reparam import Linear_model_reparam
##  from reparam.pi_dataset import Pi_dataset
##  from reparam.mini_model_deterministic import Linear_model_deterministic
##  from reparam.experiment_training_lib import run_training_loop
##  from reparam.experiment_utilise_lib import run_utilisation_loop_once, run_utilisation_loop_batch

print(torch.__version__)

In [ ]:
##import pkgutil

##for module in pkgutil.iter_modules():
##  print(module.name)

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

##  Following is essential for reparameterisation;
##    bfloat16 can cause vanishing gradient.
torch.set_default_dtype(torch.float64)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class Pi_dataset(Dataset):


    def __init__(self, filename):

        with open(filename, 'r') as f:

            line_count = sum(1 for line in f)

            print(line_count)

            self.data = torch.zeros((line_count, 1), dtype=torch.float64)

            self.labels = torch.zeros((line_count, 1), dtype=torch.float64)


        with open(filename, 'r') as f:

            for i, line in enumerate(f):

                a = line.strip().split(',')

                self.data[i] = float(a[0])

                self.labels[i] = float(a[1])


    def __len__(self):

        return len(self.data)


    def __getitem__(self, idx):

        return self.data[idx], self.labels[idx]

In [ ]:
class Linear_bayesian(nn.Module):


    def __init__(self, in_features, out_features, w_mu_init = -0.05, b_mu_init = -0.05, w_rho_init = -3.0, b_rho_init = -3.0):

        super().__init__()

        self.in_features = in_features

        self.out_features = out_features

        ##  Generate dynamic tensors for approximation posterior.

        self.w_mu = nn.Parameter(torch.Tensor(out_features, in_features).fill_(w_mu_init))

        self.w_rho = nn.Parameter(torch.Tensor(out_features, in_features).fill_(w_rho_init))

        self.b_mu = nn.Parameter(torch.Tensor(out_features).fill_(b_mu_init))

        self.b_rho = nn.Parameter(torch.Tensor(out_features).fill_(b_rho_init))

        ##  Generate static tensors for prior distribution.

        ##  F.softplus(rho) = log(1 + exp(rho))

        self.register_buffer(
            "prior_w_sigma",
            F.softplus(torch.Tensor(out_features, in_features).fill_(w_rho_init))
        )

        self.register_buffer(
            "prior_b_sigma",
            F.softplus(torch.Tensor(out_features, in_features).fill_(b_rho_init))
        )

        self.register_buffer(
            "prior_w_mu",
            torch.Tensor(out_features, in_features).fill_(0)
        )

        self.register_buffer(
            "prior_b_mu",
            torch.Tensor(out_features, in_features).fill_(0)
        )

        self.kl_loss = 0


    def kl(self):

        return self.kl_loss


    def forward(self, x):

        ##  Setup distribution based on sigma values.
        ##  Despite being static in value, will only be on correct
        ##  device if defined within forward() pass.

        self.prior_w_dist = dist.Normal(self.prior_w_mu, self.prior_w_sigma)

        self.prior_b_dist = dist.Normal(self.prior_b_mu, self.prior_b_sigma)

        ##  PyTorch built-ins used as much as possible, in this
        ##  implementation.

        ##  sigma = log(1 + exp(rho))

        ##  Generate sigma based on current values of rho.

        w_sigma = F.softplus(self.w_rho)

        b_sigma = F.softplus(self.b_rho)

        ##  print("Sigma summation:", w_sigma.sum())

        ##  Setup approximation function based on Parameter objects.

        q_w = dist.Normal(self.w_mu, w_sigma)

        q_b = dist.Normal(self.b_mu, b_sigma)

        ##  Sample via PyTorch official reparameterisation
        ##  implementation.

        w = q_w.rsample()

        b = q_b.rsample()

        ##  Recalculate KL divergence: KL(q || p).

        kl_w = dist.kl_divergence(q_w, self.prior_w_dist).sum()

        kl_b = dist.kl_divergence(q_b, self.prior_b_dist).sum()

        ##  Note that this final result will be based on the full
        ##  batch size.

        self.kl_loss = kl_w + kl_b

        return F.linear(x, w, b)


In [ ]:
class Linear_model_reparam(nn.Module):

    def __init__(self, n: int, mu_w_init: float, mu_b_init: float, rho_w_init: float, rho_b_init: float):

        ##  n = neurons in 1-n-n-1 network

        super().__init__()

        self.linear_relu_stack = nn.Sequential(
            Linear_bayesian(1, n, mu_w_init, mu_b_init, rho_w_init, rho_b_init),
            nn.ReLU(),
            Linear_bayesian(n, n, mu_w_init, mu_b_init, rho_w_init, rho_b_init),
            nn.ReLU(),
            Linear_bayesian(n, 1, mu_w_init, mu_b_init, rho_w_init, rho_b_init),
            nn.Sigmoid()
        )

        self.params_n = self.calc_params_n(self.linear_relu_stack)

        self.kl_m = 0

        ##  Non-probabilistic equivalent:

        # self.linear_relu_stack = nn.Sequential(
        #     nn.Linear(1, n),
        #     nn.ReLU(),
        #     nn.Linear(n, n),
        #     nn.ReLU(),
        #     nn.Linear(n, 1)
        # )

        # nn.init.uniform_(self.linear_relu_stack[0].weight, a=-0.25, b=0.25)
        # nn.init.uniform_(self.linear_relu_stack[2].weight, a=-0.25, b=0.25)
        # nn.init.uniform_(self.linear_relu_stack[4].weight, a=-0.25, b=0.25)


    def forward(self, x: torch.Tensor):

        y_hat = self.linear_relu_stack(x)

        linear_layers = tuple(filter(
            lambda m: isinstance(m, Linear_bayesian), self.linear_relu_stack.modules()
        ))

        kl_layers = tuple(map(lambda l: l.kl(), linear_layers))

        kl_model = sum(kl_layers) / self.params_n

        self.kl_m = kl_model

        return y_hat, kl_model


    def calc_params_n(self, model: nn.Module):

        ##  Takes a model, and finds |θ|, that is, the quantity of weights
        ##  and biases across all layers.

        ##  Extract layers.

        linear_layers = tuple(filter(
            lambda x: isinstance(x, Linear_bayesian), model.modules()
        ))

        ##  Extract neurons in to each layer.

        features = tuple(map(lambda m: m.in_features, linear_layers))

        ##  Setup pairs of adjacent layers.

        pairs = list(map(lambda i: (features[i], features[i+1]), range(len(features)-1)))

        ##  Multiply each pair.

        products = list(map(lambda pair: pair[0] * pair[1], pairs))

        return sum(products)

In [ ]:
class Linear_model_deterministic(nn.Module):

    def __init__(self, n: int, udist: tuple[int, int], random_seed: int):

        super().__init__()

        self.linear_relu_stack = nn.Sequential(
            nn.Linear(1, n),
            nn.ReLU(),
            nn.Linear(n, n),
            nn.ReLU(),
            nn.Linear(n, 1)
        )

        nn.init.uniform_(self.linear_relu_stack[0].weight, a=-udist[0], b=udist[1])
        nn.init.uniform_(self.linear_relu_stack[2].weight, a=-udist[0], b=udist[1])
        nn.init.uniform_(self.linear_relu_stack[4].weight, a=-udist[0], b=udist[1])


    def forward(self, x: torch.Tensor):

        logits = self.linear_relu_stack(x)

        out = torch.sigmoid(logits)

        return out, 0

In [ ]:
def run_model(env):

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    ##  Setup filename to store params to.

    filename = f"model_params_batch_{env['batch_n']}_epochs_{env['training_epochs']}_{env['params_name']}"

    file_path = os.path.join(*["/content", "drive", "MyDrive", filename])

    print(file_path)


    ##  Define model.

    if env["udist"] == (0, 0):

        model = Linear_model_reparam(
                    env["neurons_n"],
                    env["mu_w_init"],
                    env["mu_b_init"],
                    env["rho_w_init"],
                    env["rho_b_init"],
        ).to(device)

    else:

        model = Linear_model_deterministic(
                    env["neurons_n"],
                    env["udist"],
                    env["random_seed"],
        ).to(device)

    ##  Ensure multiple GPUs used if available.

    if torch.cuda.device_count() > 1:

        print(f"Using {torch.cuda.device_count()} GPUs!")

        model = nn.DataParallel(model, device_ids=[0, 1])

    ##  Either load params or load training data.

    print(env)

    if env["params_load"] is not None:

        run_utilisation_loop_once(
                    model,
                    env["params_load"],
                    env["params_load"].split('/')[-1] + "_plot",
                    env["sample_quantity"]
        )

    elif env["params_load_batch"] is not None:

        run_utilisation_loop_batch(
                    model,
                    env["params_load_batch"],
                    env["sample_quantity"]
        )

    else:

        train_dl = DataLoader(
                    Subset(
                        Pi_dataset("/content/drive/MyDrive/modules_work/COMP30040/datasets/pi_dataset_40000.txt"),
                        range(40000)
                    ),
                    batch_size = env["batch_n"],
                    shuffle=True
        )

        test_dl = DataLoader(
                    Subset(
                        Pi_dataset("/content/drive/MyDrive/modules_work/COMP30040/datasets/pi_dataset_8000.txt"),
                        range(8000)
                    ),
                    batch_size = env["batch_n"],
                    shuffle=True
        )

        if env["params_checkpoint"] is not None:

            print("Loading checkpoint.")

            torch.load(env["params_checkpoint"])

        run_training_loop(model,
                          train_dl,
                          test_dl,
                          env["training_epochs"],
                          file_path,
                          env["kl_loss_ratio"])


In [ ]:
def run_training_loop(model: nn.Module, train_dl: DataLoader, test_dl: DataLoader, epochs: int, filename: str,
                      kl_loss_ratio: float):

    ##  Define loss.

    loss = nn.BCELoss()

    ##  Define parameter optimisation mechanism.

    ##  optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # while time.time() - start_time < 300:

    ##  Train either for fixed period of time or fixed quantity of epochs.

    start_time = time.time()
    ##  Output training start time.

    print(datetime.now().strftime("%H:%M:%S"))

    ##  Run train-test sequence.

    for t in range(epochs):

        print(f"Training...")

        train(train_dl, model, loss, optimizer, kl_loss_ratio)

        print(f"Testing...")

        test(test_dl, model, loss, kl_loss_ratio)

    ##  Output training time.

    print(f"{time.time() - start_time}s")

    # print(model.state_dict())

    ##  Save model params to plaintext.

    # with open("model_params.post", "w") as f:

    #     state_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()

    #     f.write(str(state_dict))

    ##  Generate timestamp and store weights for later loading back.

    print(filename)

    torch.save(model.state_dict(), filename)


def train(dl: DataLoader, model: nn.Module, loss: nn.Module,
          optimizer: torch.optim.Optimizer,
          kl_loss_ratio: float):

    ##  Put model into training mode; ensures gradient tracking.

    model.train()

    for batch, (X, y) in enumerate(dl):

        X, y = X.to(device), y.to(device)

        ##  Run X through model, generate prediction.

        y_hat, kl_model = model(X)

        ##  Calculate error of prediction.

        loss_res = (1 - kl_loss_ratio) * loss(y_hat, y) + (kl_loss_ratio * kl_model)

        ##  loss_res = loss(y_hat, y)


        ##  compute gradients numerically via backpropagation, back to
        ##  leaf nodes of graph.

        loss_res.backward()

        ##  Iterate parameters.

        optimizer.step()

        ##  Reset gradients within graph.

        optimizer.zero_grad()

        # optimizer.debug_off()

        if batch % 1000 == 0:

            pass

            # print(torch.stack([X, y, y_hat], dim=0).T)

            ##  It is not necessary to know the exact parameter
            ##  values, just that they are changing.

            # optimizer.debug_off()

            # print(model.state_dict().keys())

            # print(model.state_dict()['linear_relu_stack.0.weight_mu'].sum(),
            #       model.state_dict()['linear_relu_stack.0.rho_weight'].sum(),
            #       model.state_dict()['linear_relu_stack.0.mu_bias'].sum(),
            #       model.state_dict()['linear_relu_stack.0.rho_bias'].sum(),
            #       model.state_dict()['linear_relu_stack.2.weight_mu'].sum(),
            #       model.state_dict()['linear_relu_stack.2.rho_weight'].sum(),
            #       model.state_dict()['linear_relu_stack.2.mu_bias'].sum(),
            #       model.state_dict()['linear_relu_stack.2.rho_bias'].sum(),
            #       model.state_dict()['linear_relu_stack.4.weight_mu'].sum(),
            #       model.state_dict()['linear_relu_stack.4.rho_weight'].sum(),
            #       model.state_dict()['linear_relu_stack.4.mu_bias'].sum(),
            #       model.state_dict()['linear_relu_stack.4.rho_bias'].sum(),
            #       )

            # print(model.state_dict()['linear_relu_stack.0.weight_mu'][0])
            # print(model.state_dict()['linear_relu_stack.0.weight_mu'][0].grad)

            # print(torch.round(y_hat), y)

            # print(loss_res.item())

            # current = (batch + 1) * len(X)

            # print(f"[{current:>5d}/{len(train_dl.dataset):>5d}]")

            # print(f"loss_1: {loss_1:>7f}  [{current:>5d}/{size:>5d}]")


def test(dl: DataLoader, model: nn.Module, loss: nn.Module, kl_loss_ratio: float):

    ##  Setup parameters for loss function.

    model.eval()

    test_loss, correct = 0, 0

    with torch.no_grad():

        for X, y in dl:

            X, y = X.to(device), y.to(device)

            y_hat, kl_model = model(X)

            ##  Calculate error of prediction.

            print("BCE: ", loss(y_hat, y))

            print("KL: ", kl_model)

            loss_res = (1 - kl_loss_ratio) * loss(y_hat, y) + (kl_loss_ratio * kl_model)

            test_loss += loss_res.item()

            # correct += (y_hat.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= len(dl)

    # correct /= len(dl.dataset)

    print(f"{test_loss:>8f}")

In [ ]:
def run_utilisation_loop_batch(model: nn.Module, batch_name: str, sample_quantity: int):

    print("loop batch")

    directory = Path('./params/')

    # Find all files with batch_name in filename

    for file in directory.glob(f"model_params*{batch_name}*xz"):

        print(str(file))

        if file.is_file():

            dist_metadata = str(file).split(batch_name)[-1].replace('.xz','')

            graph_filename = f"{batch_name}{dist_metadata}"

            params_path = f"./{str(file)}"

            run_utilisation_loop_once(model, params_path, graph_filename)

            # Do something with the file


def run_utilisation_loop_once(model: nn.Module, params_path: str, graph_filename: str, sample_quantity: int):

    ##  Decompress to memory.

    # with lzma.open(params_path, 'rb') as f:

    #     decompressed_data = f.read()

    ##  Load params.

    # checkpoint = torch.load(io.BytesIO(decompressed_data),

    checkpoint = torch.load(params_path,
                           weights_only = True,
                           map_location=torch.device('cpu')
                           )

    old_state_dict = checkpoint['state_dict'] if 'state_dict' in checkpoint else checkpoint

    print(old_state_dict)

    ##  Fix naming scheme and load module.

    model.load_state_dict(
                {k.replace('module.', '', 1): v for k, v in old_state_dict.items()}
    )

    model.eval()

    res = []

    interval = 1 / (2 ** 6)

    dl = DataLoader(
                TensorDataset(
                    torch.arange(1.57, 4.71 + interval, interval)
                ),
                batch_size = 1,
                shuffle = False
    )

    with torch.no_grad():

            ##################  Deterministic single-sample loop.

            # outputs_set = []

            # for batch, (X,) in enumerate(dl):

            #     X = X.to(device)

            #     # print(X)

            #     ##  Run X through model, generate prediction.

            #     y_hat, _ = model(X)

            #     print(X.item(), y_hat.item())

            #     outputs_set.append((X.item(), y_hat.squeeze(0).item()))

            ##################  Reparam multi-sample loop.

            outputs_set = ()

            for batch, (X,) in enumerate(dl):

                X = X.to(device)

                # print(X)

                ##  Run X through model, generate prediction.

                sample_set = ()

                for _ in range(sample_quantity):

                    y_hat_sample, _ = model(X)

                    sample_set = sample_set + (y_hat_sample,)

                print(sample_set)

                ##  Take mean of samples.

                y_hat = torch.stack(sample_set).mean(dim=0)

                print(f"p({X.item()}) = {y_hat.item()}")

                ##  Store for later plotting

                outputs_set = outputs_set + ((X.item(), y_hat.squeeze(0).item()),)

            ##################

            x, y = zip(*outputs_set)

            plt.figure(figsize=(6, 4))

            plt.plot(
                        x, y,
                        marker='o',
                        markersize=4,          # small dots
                        markerfacecolor='black',
                        markeredgecolor='black',
                        linewidth=0.8,         # thin connecting line
                        color='black',
                        label='Trajectory'
            )

            # plt.scatter(
            #             x, y,
            #             color='black',          # point colour
            #             edgecolor='black',          # optional border around each marker
            #             s=10,                       # marker size
            #             label='Samples'
            # )

            plt.xlabel('Input value')
            plt.ylabel('Probability of π classification')
            plt.grid(True, ls='--', lw=0.5, alpha=0.7)
            plt.tight_layout()

            plt.tight_layout()

            plt.savefig(f"{graph_filename}.pdf", dpi=300)
            #plt.show()

            plt.close()

            return graph_filename



In [13]:
# env_train_det = {
#             "neurons_n": 1024,
#             "batch_n": 64,
#             "training_epochs": 50,
#             "params_name": "det",       ##  Filename to store params to.
#             "params_load": None,        ##  Filename to load specific set of params from.
#             "params_load_batch": None,  ##  Filename to wildcard match a multiple sets of params.
#             "random_seed": 239852,
#             "mu_w_init": 0.0,
#             "mu_b_init": 0.0,
#             "rho_w_init": 0.0,
#             "rho_b_init": 0.0,
#             "kl_loss_ratio": 0.0,       ##  Ratio to consider KL-divergence loss during sum of loss functions.
#             "udist": (0.1, 0.1)         ##  Uniform distribution params, in the case of deterministic model.
# }

env_train_reparam = {
            "neurons_n": 1024,
            "batch_n": 16,
            "training_epochs": 35,
            "params_name": "reparam_adam_kl",   ##  Filename to store params to.
            "params_load": None,        ##  Filename to load specific set of params from, for utilisation.
            "params_load_batch": None,  ##  Filename to wildcard match a multiple sets of params.
            "params_checkpoint": None, ##"/content/drive/MyDrive/model_params_batch_16_epochs_34_reparam",
            "random_seed": 239852,
            "mu_w_init": 0.1,
            "mu_b_init": -3.0,
            "rho_w_init": 0.1,
            "rho_b_init": -3.0,
            "kl_loss_ratio": 0.5,       ##  Ratio to consider KL-divergence loss during sum of loss functions.
            "udist": (0, 0)             ##  Uniform distribution params, if not (0, 0) will trigger deterministic model.
}

#for k, v in env.items():
#            print(f"{k}: {v}")

torch.manual_seed(env_train_reparam["random_seed"])

run_model(env_train_reparam)

Streaming output truncated to the last 5000 lines.
KL:  tensor(0.0814, device='cuda:0')
BCE:  tensor(62.5000, device='cuda:0')
KL:  tensor(0.0814, device='cuda:0')
BCE:  tensor(50., device='cuda:0')
KL:  tensor(0.0814, device='cuda:0')
BCE:  tensor(62.5000, device='cuda:0')
KL:  tensor(0.0814, device='cuda:0')
BCE:  tensor(56.2500, device='cuda:0')
KL:  tensor(0.0814, device='cuda:0')
BCE:  tensor(62.5000, device='cuda:0')
KL:  tensor(0.0814, device='cuda:0')
BCE:  tensor(43.7500, device='cuda:0')
KL:  tensor(0.0814, device='cuda:0')
BCE:  tensor(37.5000, device='cuda:0')
KL:  tensor(0.0814, device='cuda:0')
BCE:  tensor(31.2500, device='cuda:0')
KL:  tensor(0.0814, device='cuda:0')
BCE:  tensor(31.2500, device='cuda:0')
KL:  tensor(0.0814, device='cuda:0')
BCE:  tensor(43.7500, device='cuda:0')
KL:  tensor(0.0814, device='cuda:0')
BCE:  tensor(37.5000, device='cuda:0')
KL:  tensor(0.0814, device='cuda:0')
BCE:  tensor(37.5000, device='cuda:0')
KL:  tensor(0.0814, device='cuda:0')
BCE:

In [14]:
filename_det = f"/content/drive/MyDrive/model_params_batch_{env_train_det['batch_n']}_epochs_{env_train_det['training_epochs']}_{env_train_det['params_name']}"

env_load_det = {
            "params_name": None,
            "params_load": filename_det,
            "params_load_batch": None,
            "sample_quantity": 10
}

print(filename_det)

env_utilise_det = env_train_det | env_load_det

NameError: name 'env_train_det' is not defined

In [ ]:
filename_reparam = f"/content/drive/MyDrive/model_params_batch_{env_train_reparam['batch_n']}_epochs_{env_train_reparam['training_epochs']}_{env_train_reparam['params_name']}"

env_load_reparam = {
            "params_name": None,
            "params_load": filename_reparam,
            "params_load_batch": None,
            "sample_quantity": 50
}

env_utilise_reparam = env_train_reparam |  env_load_reparam

for k, v in env_utilise_reparam.items():
            print(f"{k}: {v}")

run_model(env_utilise_reparam)

In [ ]:
loss_f_out = [
    42.696472,
41.477603,
41.014276,
40.667021,
40.932900,
40.575959,
38.571851,
38.918440,
40.359959,
40.460665,
38.799338,
38.219441,
37.202821,
37.102280,
36.488305,
36.460563,
35.876466,
35.944494,
33.299981,
33.017012,
33.439892,
32.360254,
31.752009,
30.638090,
29.561501,
29.059143,
26.966269,
26.435792,
25.272441,
24.816166,
23.768201,
22.820384,
21.245264,
17.784418,
5.418300,
2.576705,
1.561233,
1.142938,
0.971437,
0.895156,
0.805152,
0.815637,
0.888525,
0.782627,
0.756303,
0.722468,
0.731347,
0.736260,
0.755868,
0.803535,
0.663457,
0.699057,
0.658999,
0.697714,
0.709434,
0.712090,
0.686332,
0.768111,
0.651804,
0.699404,
0.701798,
0.744775,
0.692163,
0.638962,
0.679666,
0.658388,
0.655789,
0.664435,
0.666605,
0.671217,
0.652489,
0.663480,
0.663125,
0.657119,
0.648425,
0.653090,
0.676391,
0.655319,
0.712031,
0.669462,
0.658036,
0.663246,
0.732317,
0.694492,
0.691628,
0.659758,
0.702757,
0.658387,
0.672982,
0.716305,
0.684409,
0.653400,
0.658829,
0.652619,
0.691526,
0.670583,
0.637470,
0.674577,
0.683531,
0.670200
]

# 2. Generate ascending integer values for the x-axis
# This creates a sequence from 0 to 9 based on the data length
x_values = range(len(loss_f_out))

# 3. Create the plot
plt.plot(x_values, loss_f_out, marker='o', markersize=1.5, linestyle='-', linewidth=0.5, color='black')

# 4. Add labels and title
plt.xlabel('Training epoch')
plt.ylabel('Loss function output (logarithmic)')

plt.tight_layout()

plt.yscale('log')

plt.grid(True, which="both", ls="-", alpha=0.5)

#plt.show()

plt.savefig('loss_func_out_per_epoch.pdf')